In [1]:
import numpy as np
import cv2
import pickle
import os

# Load Phase 1 results
with open('phase1_results.pkl', 'rb') as f:
    p1 = pickle.load(f)

img = p1['img']
height, width = p1['height'], p1['width']
region_map = p1['region_map']
region_id = p1['region_id']
rsizes = p1['rsizes']

output_dir = "comic/phase2_output"
os.makedirs(output_dir, exist_ok=True)

print(f"Phase 1 data loaded")
print(f"Image: {height}x{width}, Regions: {region_id}")

# Save original image
original_path = os.path.join(output_dir, "00_original_image.jpg")
cv2.imwrite(original_path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))

Phase 1 data loaded
Image: 393x800, Regions: 38238


True

In [2]:
def detect_text_regions(img, region_map, rsizes, max_size=150):
    """
    Detect text regions using Otsu's automatic threshold, combined with
    a size cap only. Unlike edge/outline strokes (uniformly thin),
    individual letters have mixed shapes — thin stems and rounded,
    blob-like bowls (e.g. 'o', 'e', 'a') — so an elongation/shape filter
    would incorrectly exclude the rounded parts of letters. Clothing and
    other large solid-color regions are reliably excluded by size alone,
    since they span thousands of pixels versus tens to ~150 pixels for
    a letter or letter fragment.
    """
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    otsu_thresh, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    print(f"Otsu-determined threshold: {otsu_thresh:.1f}")

    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, iterations=1)

    text_regions = set()
    unique_rids = np.unique(region_map[region_map >= 0])

    for rid in unique_rids:
        if not (0 < rsizes[rid] < max_size):
            continue

        region_mask = (region_map == rid)
        total = np.sum(region_mask)
        if total == 0:
            continue

        text_ratio = np.sum(binary[region_mask] > 0) / total
        if text_ratio > 0.5:
            text_regions.add(int(rid))

    return text_regions, otsu_thresh

text_regions, otsu_thresh = detect_text_regions(img, region_map, rsizes)
print(f"Text regions detected: {len(text_regions)}")

Otsu-determined threshold: 155.0
Text regions detected: 11385


In [3]:
def detect_edge_regions_by_shape(region_map, rsizes, img, max_size=100, min_aspect_ratio=2.5, max_extent=0.5):
    """
    Detect thin edge/outline regions using shape characteristics instead
    of only size and brightness. Text strokes and outlines are thin and
    elongated: high aspect ratio (long relative to width) and low extent
    (fill ratio = region area / bounding box area, since a thin curved
    stroke occupies a small fraction of its bounding box).

    This distinguishes genuine thin strokes from small but blob-like
    dark regions (e.g. shadow fragments, dark clothing patches), which
    tend to have low aspect ratio and high extent.
    """
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    edge_regions = set()
    unique_rids = np.unique(region_map[region_map >= 0])

    for rid in unique_rids:
        if not (0 < rsizes[rid] < max_size):
            continue

        mask = (region_map == rid).astype(np.uint8)
        ys, xs = np.where(mask > 0)
        if len(ys) == 0:
            continue

        # Bounding box dimensions
        h = ys.max() - ys.min() + 1
        w = xs.max() - xs.min() + 1
        bbox_area = h * w
        region_area = len(ys)

        aspect_ratio = max(h, w) / max(min(h, w), 1)
        extent = region_area / bbox_area  # fill ratio within bounding box

        avg_brightness = np.mean(gray[mask > 0])

        is_elongated = aspect_ratio >= min_aspect_ratio
        is_sparse_fill = extent <= max_extent
        is_dark = avg_brightness < 150

        if is_dark and (is_elongated or is_sparse_fill):
            edge_regions.add(int(rid))

    return edge_regions

edge_regions = detect_edge_regions_by_shape(region_map, rsizes, img)
print(f"Edge/outline regions detected (shape-based): {len(edge_regions)}")

special_regions = text_regions | edge_regions
print(f"Total special regions (text + edge): {len(special_regions)}")

special_px = sum(int(np.sum(region_map == rid)) for rid in special_regions)
print(f"Special region pixels: {special_px} ({special_px / (height*width) * 100:.2f}% of image)")

Edge/outline regions detected (shape-based): 3039
Total special regions (text + edge): 11387
Special region pixels: 54942 (17.48% of image)


In [4]:
phase2_results = {
    'img': img,
    'height': height,
    'width': width,
    'region_map': region_map,
    'text_regions': text_regions,
    'edge_regions': edge_regions,
    'special_regions': special_regions
}

with open('phase2_results.pkl', 'wb') as f:
    pickle.dump(phase2_results, f)

print("Phase 2 results saved to phase2_results.pkl")
print(f"Key outputs:")
print(f"  - text_regions: {len(text_regions)}")
print(f"  - edge_regions: {len(edge_regions)}")
print(f"  - Total special regions: {len(special_regions)}")

Phase 2 results saved to phase2_results.pkl
Key outputs:
  - text_regions: 11385
  - edge_regions: 3039
  - Total special regions: 11387


In [5]:

# Text regions 
text_vis = img.copy().astype(float)
for rid in text_regions:
    mask = (region_map == rid)
    text_vis[mask] = [255, 0, 0]  

text_vis_path = os.path.join(output_dir, "01_text_regions_RED.jpg")
cv2.imwrite(text_vis_path, cv2.cvtColor(text_vis.astype(np.uint8), cv2.COLOR_RGB2BGR))

# Text + Edge 
special_vis = img.copy().astype(float)
for rid in special_regions:
    mask = (region_map == rid)
    special_vis[mask] = [255, 0, 0]  

special_vis_path = os.path.join(output_dir, "02_text_and_edge_RED.jpg")
cv2.imwrite(special_vis_path, cv2.cvtColor(special_vis.astype(np.uint8), cv2.COLOR_RGB2BGR))

print("Red visualization saved")

Red visualization saved
